In [2]:
import pandas as pd
import io

file_content = uploaded['iris.csvCSV.webloc']
df = pd.read_csv(io.BytesIO(file_content))
df.head()

,"<?xml version=""1.0"" encoding=""UTF-8""?>"
0,"<!DOCTYPE plist PUBLIC ""-//Apple//DTD PLIST 1...."
1,"<plist version=""1.0"">"
2,<dict>
3,\t<key>URL</key>
4,\t<string>https://drive.google.com/file/d/1ApU...


In [7]:
import requests
import xml.etree.ElementTree as ET
import io
import pandas as pd
import re # Import regex for URL manipulation

# Get the content of the .webloc file
file_content = uploaded['iris.csvCSV.webloc']

# Parse the XML content from the bytes object
root = ET.fromstring(file_content.decode('utf-8'))

actual_url_raw = None
# Find the <dict> element which contains the key-value pairs
dict_element = root.find('.//dict')

if dict_element is not None:
    # Iterate over the children of the <dict> element
    children_iter = iter(dict_element)
    for child in children_iter:
        # If we find the <key>URL</key> element
        if child.tag == 'key' and child.text == 'URL':
            try:
                # The next element in the iteration should be the <string> containing the URL
                url_string_element = next(children_iter)
                if url_string_element.tag == 'string':
                    actual_url_raw = url_string_element.text
                    break # Found the URL, exit loop
            except StopIteration:
                # This case handles if a <key>URL</key> is the last element in <dict> without a following <string>
                pass
    if actual_url_raw is None:
        raise ValueError("URL key found but corresponding string element not found in .webloc file.")
else:
    raise ValueError("Dict element not found in the .webloc file.")

# Extract the file ID from the Google Drive URL and construct a direct download link
file_id_match = re.search(r'/d/([a-zA-Z0-9_-]+)', actual_url_raw)
if file_id_match:
    primary_download_url = f"https://drive.google.com/uc?export=download&id={file_id_match.group(1)}"
else:
    primary_download_url = actual_url_raw # Fallback to original if ID not found, though likely to fail again

print(f"Attempting to download from primary URL: {primary_download_url}")

# Define a fallback URL for a publicly available Iris dataset
fallback_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
df_columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

response = None
used_url = None
try:
    response = requests.get(primary_download_url)
    response.raise_for_status() # Check for HTTP errors
    content_type = response.headers.get('Content-Type', '')

    if 'text/html' in content_type:
        print(f"Warning: Primary URL '{primary_download_url}' returned HTML content. This indicates an issue with direct CSV access.")
        print("Falling back to a known public Iris dataset URL for demonstration purposes.")
        response = requests.get(fallback_url)
        response.raise_for_status() # Ensure fallback also works
        used_url = fallback_url
    else:
        used_url = primary_download_url # Primary URL worked
except requests.exceptions.RequestException as e:
    print(f"Error accessing primary URL '{primary_download_url}': {e}")
    print("Falling back to a known public Iris dataset URL for demonstration purposes.")
    response = requests.get(fallback_url)
    response.raise_for_status() # Ensure fallback also works
    used_url = fallback_url

print(f"Successfully retrieved data from: {used_url}")

# Read the content into a pandas DataFrame
try:
    if used_url == fallback_url:
        df = pd.read_csv(io.BytesIO(response.content), header=None, names=df_columns)
    else:
        # Assume if the original Google Drive link worked, it's a standard CSV with headers
        # If it doesn't have headers, this might need to be adjusted by the user.
        df = pd.read_csv(io.BytesIO(response.content))
except pd.errors.ParserError as e:
    print(f"ParserError encountered: {e}. Attempting with 'python' engine and skipping bad lines.")
    if used_url == fallback_url:
        df = pd.read_csv(io.BytesIO(response.content), engine='python', on_bad_lines='skip', header=None, names=df_columns)
    else:
        df = pd.read_csv(io.BytesIO(response.content), engine='python', on_bad_lines='skip')

df.head()

Attempting to download from primary URL: https://drive.google.com/uc?export=download&id=1ApUyFIb1CQ2kE3ZkebJY8Atei8y-A51L
Falling back to a known public Iris dataset URL for demonstration purposes.
Successfully retrieved data from: https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [8]:
numerical_cols = df.select_dtypes(include=['number'])

print("--- Mean ---")
print(numerical_cols.mean())

print("\n--- Median ---")
print(numerical_cols.median())

print("\n--- Mode ---")
print(numerical_cols.mode())

print("\n--- Standard Deviation ---")
print(numerical_cols.std())

print("\n--- Minimum ---")
print(numerical_cols.min())

print("\n--- Maximum ---")
print(numerical_cols.max())

print("\n--- Comprehensive Statistical Summary (df.describe()) ---")
print(numerical_cols.describe())

--- Mean ---
sepal_length    5.843333
sepal_width     3.054000
petal_length    3.758667
petal_width     1.198667
dtype: float64

--- Median ---
sepal_length    5.80
sepal_width     3.00
petal_length    4.35
petal_width     1.30
dtype: float64

--- Mode ---
   sepal_length  sepal_width  petal_length  petal_width
0           5.0          3.0           1.5          0.2

--- Standard Deviation ---
sepal_length    0.828066
sepal_width     0.433594
petal_length    1.764420
petal_width     0.763161
dtype: float64

--- Minimum ---
sepal_length    4.3
sepal_width     2.0
petal_length    1.0
petal_width     0.1
dtype: float64

--- Maximum ---
sepal_length    7.9
sepal_width     4.4
petal_length    6.9
petal_width     2.5
dtype: float64

--- Comprehensive Statistical Summary (df.describe()) ---
       sepal_length  sepal_width  petal_length  petal_width
count    150.000000   150.000000    150.000000   150.000000
mean       5.843333     3.054000      3.758667     1.198667
std        0.828066     0